# Quantum Computing with Qiskit — Day 1

**Learning outcomes**

Today we:
1. Build circuits from **basic gates**.
2. Meet the **Bell state** and understand **entanglement**.
3. **Transpile** that Bell state for **two different backends**, watching the transpiler's stages *step by step, side by side*.
4. **Sample** the Bell state and see how measurement **accuracy improves with the number of shots**.

**Prerequisites:** Python, basic linear algebra (vectors, matrices, tensor/Kronecker products), basic probability.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

import qiskit
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, Operator
from qiskit.primitives import StatevectorSampler
from qiskit.visualization import plot_histogram, plot_bloch_multivector, plot_coupling_map
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.circuit.random import random_circuit

print("Qiskit version:", qiskit.__version__)

import os, sys
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]

import shutil
dot = shutil.which("dot")
if dot:
    print("Graphviz binary found:", dot)
else:
    print("Graphviz binary NOT found")
    print("Pass-manager diagrams won't render. Install it (see table above), then restart the kernel.")

## 1. Building circuits from basic gates

### Quantum states and amplitudes

A single qubit lives in a 2-dimensional complex vector space. The computational basis states are $|0\rangle = \begin{pmatrix}1\\0\end{pmatrix}$ and $|1\rangle = \begin{pmatrix}0\\1\end{pmatrix}$.

A general single-qubit state can be written as:
$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle = \begin{pmatrix}\alpha\\\beta\end{pmatrix}$$

where $\alpha, \beta \in \mathbb{C}$ are complex amplitudes satisfying the normalization condition $|\alpha|^2 + |\beta|^2 = 1$.

When we measure this qubit in the computational basis:
- We get outcome $|0\rangle$ with probability $|\alpha|^2$
- We get outcome $|1\rangle$ with probability $|\beta|^2$

This is the **Born rule**: measurement probabilities come from the squared magnitudes of amplitudes.

### The Hadamard gate

The Hadamard gate is one of the most important single-qubit gates. It creates **superposition** — a quantum state that is simultaneously in multiple classical states.

The Hadamard gate $H$ has the matrix representation:
$$H = \frac{1}{\sqrt{2}}\begin{pmatrix}1 & 1\\1 & -1\end{pmatrix}$$

When applied to the basis states:
- $H|0\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle) = |+\rangle$ — equal superposition
- $H|1\rangle = \frac{1}{\sqrt{2}}(|0\rangle - |1\rangle) = |-\rangle$ — equal superposition with opposite phase

The Hadamard is its own inverse: $H^2 = I$, so applying it twice returns you to the original state.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)                    # Hadamard: puts |0> into an equal superposition
qc.draw("mpl")

### Visualizing quantum states

Let's verify what the Hadamard gate does by looking at the state vector:

In [ ]:
# Create the state |+> = H|0>
qc = QuantumCircuit(1)
qc.h(0)
state = Statevector(qc)
print("State vector after Hadamard:")
print(state)
print(f"\nAmplitudes: α = {state.data[0]:.3f}, β = {state.data[1]:.3f}")
print(f"Probabilities: |α|² = {abs(state.data[0])**2:.3f}, |β|² = {abs(state.data[1])**2:.3f}")

In [ ]:
state = Statevector(qc)
print("Amplitudes [alpha, beta]:", state.data)
print("Probabilities |amp|^2   :", state.probabilities())

$H|0\rangle=\tfrac1{\sqrt2}(|0\rangle+|1\rangle)$: each amplitude is $1/\sqrt2\approx0.707$, each probability $1/2$.

### Single-qubit gates are $2\times2$ unitary matrices
A gate acts by matrix multiplication, $|\psi'\rangle=U|\psi\rangle$, and unitarity ($U^\dagger U=I$) keeps the state normalized.

| Gate | Matrix | Effect |
|------|--------|--------|
| $X$ | $\begin{pmatrix}0&1\\1&0\end{pmatrix}$ | bit-flip ("NOT") |
| $Z$ | $\begin{pmatrix}1&0\\0&-1\end{pmatrix}$ | phase-flip |
| $H$ | $\tfrac1{\sqrt2}\begin{pmatrix}1&1\\1&-1\end{pmatrix}$ | make/undo superposition |

In [ ]:
def gate_matrix(builder):
    c = QuantumCircuit(1); builder(c)
    return Operator(c).data

print("X =\n", gate_matrix(lambda c: c.x(0)))
print("\nZ =\n", gate_matrix(lambda c: c.z(0)))
print("\nH =\n", gate_matrix(lambda c: c.h(0)))

### Rotation gates: a continuous knob
$R_y(\theta)$ dials any probability you like:

$$
R_y(\theta) = e^{-i\theta Y/2} =
\begin{pmatrix}
\cos\frac{\theta}{2} & -\sin\frac{\theta}{2} \\
\sin\frac{\theta}{2} & \cos\frac{\theta}{2}
\end{pmatrix}
$$

$$R_y(\theta)|0\rangle=\cos\tfrac\theta2|0\rangle+\sin\tfrac\theta2|1\rangle\;\Rightarrow\;\Pr[1]=\sin^2\tfrac\theta2.$$

$$
R_y(\theta)\,|1\rangle = -\sin\tfrac{\theta}{2}\,|0\rangle + \cos\tfrac{\theta}{2}\,|1\rangle
$$


In [ ]:
theta = np.pi/3
qc = QuantumCircuit(1); qc.ry(theta, 0)
print(f"Pr[1] = sin^2(theta/2) = {np.sin(theta/2)**2:.4f}")
print(f"Qiskit agrees       : {Statevector(qc).probabilities()[1]:.4f}")

### Many qubits: the tensor product, and the CNOT gate
Two qubits live in $\mathbb{C}^2\otimes\mathbb{C}^2=\mathbb{C}^4$ with basis $|00\rangle,|01\rangle,|10\rangle,|11\rangle$; $n$ qubits need a length-$2^n$ vector (this exponential size is why quantum systems are hard to simulate).

$$
\text{CNOT} =
\begin{pmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & 0 & 1 \\
0 & 0 & 1 & 0
\end{pmatrix}
$$

The key two-qubit gate is **CNOT** (`cx`): it flips the *target* iff the *control* is $|1\rangle$.

$$
|00\rangle \mapsto |00\rangle, \quad
|01\rangle \mapsto |01\rangle, \quad
|10\rangle \mapsto |11\rangle, \quad
|11\rangle \mapsto |10\rangle
$$

**A note on Qiskit bit ordering:** Qiskit is **little-endian** — in a string like `'10'` the **right**most char is qubit 0. So `'10'` means $q_1{=}1,q_0{=}0$. Keep this in mind for every histogram.

In [ ]:
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)     # control q0, target q1
qc.draw("mpl")

Keep this exact circuit in mind — **`H` then `CX`** is the recipe for the Bell state, our running example for the rest of the day.

## 2. The Bell state and entanglement

### Separable vs entangled
A two-qubit state is **separable** if it factors as $|\psi\rangle=|a\rangle\otimes|b\rangle$. If no such factorization exists, it is **entangled**.

The **Bell state**
$$|\Phi^+\rangle=\tfrac1{\sqrt2}\big(|00\rangle+|11\rangle\big)$$
is entangled. *Proof:* a product $(\alpha|0\rangle+\beta|1\rangle)\otimes(\gamma|0\rangle+\delta|1\rangle)$ has $|01\rangle$-amplitude $\alpha\delta$ and $|10\rangle$-amplitude $\beta\gamma$. Killing both while keeping $|00\rangle$ and $|11\rangle$ non-zero forces a contradiction, so $|\Phi^+\rangle$ cannot be factored. $\blacksquare$

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
bell.draw("mpl")

In [ ]:
sv = Statevector(bell)
print("Statevector:", sv.data, "  (order 00, 01, 10, 11)")
sv.draw("latex")



The Bloch sphere is a geometric representation of a single-qubit state. Let's visualize some common states:

In [ ]:
# Visualize |0> state
qc_0 = QuantumCircuit(1)
state_0 = Statevector(qc_0)
print("State |0>:")
plot_bloch_multivector(state_0)

In [ ]:
# Visualize |+> state (after Hadamard)
qc_plus = QuantumCircuit(1)
qc_plus.h(0)
state_plus = Statevector(qc_plus)
print("State |+> = H|0>:")
plot_bloch_multivector(state_plus)

In [ ]:
# Visualize |1> state (after X gate)
qc_1 = QuantumCircuit(1)
qc_1.x(0)
state_1 = Statevector(qc_1)
print("State |1> = X|0>:")
plot_bloch_multivector(state_1)

### Exercise: Explore different quantum states

Try creating and visualizing the following states on the Bloch sphere:

1. **State |->**: Apply H then Z to |0> (or apply X then H to |0>)
2. **State |i>**: Apply H then S to |0> (S is the phase gate)
3. **Custom rotation**: Use `ry(θ)` with different angles (try θ = π/4, π/3, π/2)

For each state, observe:
- Where it appears on the Bloch sphere
- What the amplitudes α and β are
- What measurement probabilities you would expect

In [ ]:
# Exercise solution space - try creating the states mentioned above!
# Example for state |->
qc_minus = QuantumCircuit(1)
qc_minus.x(0)
qc_minus.h(0)
state_minus = Statevector(qc_minus)
print("State |->:")
plot_bloch_multivector(state_minus)

In [ ]:
def make_bell():
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.cx(0,1)
    return qc

for kind in ["phi+", "phi-", "psi+", "psi-"]:
    print(f"|{kind}> :", np.round(Statevector(make_bell()).data, 3), "  (00, 01, 10, 11)")

### Backend coupling map visualization

Let's visualize the connectivity of our two backends to understand how qubits can interact:

In [ ]:
# Visualize the line topology
# Define coupling maps (qubit connections)
line: list[list[int]] = [[0,1], [1,2], [2,3], [3,4]]  # Line: 0-1-2-3-4
ring = [[0,1], [1,2], [2,3], [3,4], [4,0]]  # Ring: 0-1-2-3-4-0

# Line backend
plot_coupling_map(num_qubits=5, coupling_map=line, qubit_coordinates=None)

# Ring backend
plot_coupling_map(num_qubits=5, coupling_map=ring, qubit_coordinates=None)


In [ ]:
backend_line = GenericBackendV2(5, basis_gates=["id", "rz", "sx", "x", "cx"], coupling_map=line, seed=11)
backend_ring = GenericBackendV2(5, basis_gates=["id", "rz", "sx", "x", "cz"], coupling_map=ring, seed=11)

print("Backend A: native 2-qubit gate = CX,  topology = line")
print("Backend B: native 2-qubit gate = CZ,  topology = ring")

### 3a. Step by step, side by side
Every stage is itself a small pass manager. By running them one at a time we can snapshot the circuit **after each stage** and compare the two backends.

In [ ]:
def stage_snapshots(backend, circ):
    '''Run the preset pipeline one stage at a time, returning (stage_name, circuit) after each.'''
    pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
    snaps = [("original", circ)]
    cur, ps = circ, None
    for st in pm.stages:
        spm = getattr(pm, st)
        if spm is None:                 # empty stage -> circuit unchanged
            snaps.append((st, cur)); continue
        if ps is not None:
            spm.property_set = ps        # thread the shared property set across stages
        cur = spm.run(cur)
        ps = spm.property_set
        snaps.append((st, cur))
    return snaps

bell_meas = bell.copy(); bell_meas.measure_all()
snapsA = stage_snapshots(backend_line, bell_meas)
snapsB = stage_snapshots(backend_ring, bell_meas)

def twoq(ops):    # count two-qubit gates
    return sum(v for k, v in ops.items() if k in ("cx", "cz", "ecr", "swap"))

print(f"{'stage':>12} | {'A depth':>7} {'A 2q':>5} | {'B depth':>7} {'B 2q':>5}")
print("-" * 48)
for (sa, ca), (sb, cb) in zip(snapsA, snapsB):
    print(f"{sa:>12} | {ca.depth():>7} {twoq(ca.count_ops()):>5} | "
          f"{cb.depth():>7} {twoq(cb.count_ops()):>5}")

Read down the table: for both backends **layout** and **routing** leave the circuit untouched — the Bell state has only *one* two-qubit gate, so the router can always sit its two qubits next to each other (no SWAPs needed). The real action is at **translation**, where each backend rewrites the gates into *its own* basis. Let's see that visually.

In [ ]:
def pick(snaps, name):
    return dict(snaps)[name]

checkpoints = ["original", "translation", "optimization"]
labels = ["Original (abstract)", "After translation", "Final (optimized)"]

fig, axes = plt.subplots(len(checkpoints), 2, figsize=(12, 9))
for col, (name, snaps) in enumerate([("Backend A (CX, line)", snapsA),
                                     ("Backend B (CZ, ring)", snapsB)]):
    for row, (cp, lab) in enumerate(zip(checkpoints, labels)):
        ax = axes[row][col]
        circ = pick(snaps, cp).copy()
        circ.global_phase = 0       # global phase is unobservable; hide it for a clean drawing
        circ.draw("mpl", ax=ax, idle_wires=(cp == "original"), fold=-1)
        ax.set_title(f"{name}\n{lab}  (depth {circ.depth()})", fontsize=10)
plt.tight_layout()
plt.show()

**The key lesson.** The *same* Bell state lands very differently on the two devices:

- **Backend A** has CX as a native gate, so the Bell `CX` stays as one gate; only the `H` is rewritten (into `rz`–`sx`–`rz`). Result: shallow.
- **Backend B** has only CZ natively, so the Bell `CX` must be rebuilt as `CZ` sandwiched between single-qubit gates — more gates, larger depth.

Same computation, different cost — and we never touched the algorithm. *(Routing is trivial here because Bell has a single 2-qubit gate; with more interacting qubits the router would insert SWAPs, and the two topologies would then differ there too.)*

### 3b. Looking *inside* the stages — the passes as graphs
Each stage is a group of **passes**. `.draw()` renders them as a flow graph:

- **blue box** = transformation pass (rewrites the circuit),
- **red box** = analysis pass (only measures the circuit, e.g. `Depth`, `Size`),
- **dashed oval** = an argument fed into a pass.

Here is every stage of **Backend A**'s pipeline. Backend B has the *same stage structure* but different passes inside `translation` (because its basis differs).

In [ ]:
pmA = generate_preset_pass_manager(backend=backend_line, optimization_level=1)
for stage in pmA.stages:
    sub = getattr(pmA, stage)
    if sub is None:
        continue
    display(Markdown(f"#### `{stage}` stage"))
    display(sub.draw())

Qiskit's transpiler offers different optimization levels (0-3). Let's compare how they affect a randomly generated circuit:

In [ ]:
# Compare different optimization levels for the Bell circuit
optimization_levels = [0, 1, 2, 3]
results = {}

random_circ = random_circuit(2, 10, measure=True, seed=42)

for opt_level in optimization_levels:
    pm = generate_preset_pass_manager(optimization_level=opt_level, backend=backend_line)
    transpiled = pm.run(random_circ)
    ops = transpiled.count_ops()
    results[opt_level] = {
        'circuit': transpiled,
        'depth': transpiled.depth(),
        'gates': sum(ops.values()),
        'two_qubit': sum(v for k, v in ops.items() if k in ('cx', 'cz', 'ecr'))
    }

# Display comparison
print("Optimization Level Comparison (Line Backend):")
print("="*60)
print(f"{'Level':<10} {'Depth':<10} {'Total Gates':<15} {'2Q Gates':<10}")
print("="*60)
for level in optimization_levels:
    r = results[level]
    print(f"{level:<10} {r['depth']:<10} {r['gates']:<15} {r['two_qubit']:<10}")

print("\nKey observations:")
print("- Level 0: Minimal optimization, just makes circuit executable")
print("- Level 1: Light optimization, reduces some redundancy")
print("- Level 2: Medium optimization, better gate reduction")
print("- Level 3: Heavy optimization, best gate count but slower compilation")

In [ ]:
# Visualize circuits at different optimization levels
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, level in enumerate(optimization_levels):
    circ = results[level]['circuit'].copy()
    circ.global_phase = 0
    circ.draw('mpl', ax=axes[idx], fold=-1)
    axes[idx].set_title(f"Optimization Level {level}\n"
                       f"Depth: {results[level]['depth']}, "
                       f"2Q gates: {results[level]['two_qubit']}")

plt.tight_layout()
plt.show()

## 4. Sampling the Bell state — and what shots mean

We have only *peeked* at exact statevectors so far. A real quantum computer never lets you do that: you can only **measure**, getting one bitstring and collapsing the state. To estimate the probabilities you **repeat** — each run is a **shot**.

Qiskit's tool is a **Sampler** primitive. For an ideal, laptop-only run we use `StatevectorSampler` (no noise, no hardware). The circuit must contain measurements (`measure_all`), and we pass it as a one-element list of **PUBs**.

In [ ]:
sampler = StatevectorSampler(seed=42)
counts = sampler.run([bell_meas], shots=2000).result()[0].data.meas.get_counts()
print("Counts:", counts)
_ = plot_histogram(counts, title="Bell state, 2000 shots")

### First: transpilation doesn't change the *ideal* statistics
Because we simulate without noise, the abstract Bell circuit and **both** transpiled versions produce the same distribution — transpilation preserves the computation. (On real, noisy hardware they would differ; that's a Day-2 story.)

In [ ]:
tA = transpile(bell_meas, backend=backend_line, optimization_level=1, seed_transpiler=7)
tB = transpile(bell_meas, backend=backend_ring, optimization_level=1, seed_transpiler=7)
for label, circ in [("abstract", bell_meas), ("backend A", tA), ("backend B", tB)]:
    c = StatevectorSampler(seed=0).run([circ], shots=4000).result()[0].data.meas.get_counts()
    frac = {k: round(v / 4000, 3) for k, v in sorted(c.items())}
    print(f"{label:>10}: {frac}")

### The fingerprint of entanglement
Look at each qubit **alone**: a fair coin (≈50/50). Yet the two outcomes **always agree** — we only ever see `00` or `11`, never `01`/`10`. Knowing one tells you the other with certainty. No pair of *independent* coins can do that.

In [ ]:
q0 = {"0": 0, "1": 0}; q1 = {"0": 0, "1": 0}
for bits, n in counts.items():        # bits = 'q1 q0'
    q1[bits[0]] += n
    q0[bits[1]] += n
tot = sum(counts.values())
print(f"Qubit 0 alone: 0 -> {q0['0']/tot:.1%}, 1 -> {q0['1']/tot:.1%}")
print(f"Qubit 1 alone: 0 -> {q1['0']/tot:.1%}, 1 -> {q1['1']/tot:.1%}")
print("Jointly: only 00 and 11 ever appear -> perfectly correlated.")

### What a "shot" means mathematically
Let the circuit's outcome probabilities be $p_x=|\langle x|\psi\rangle|^2$ (Born rule); for our Bell state $p_{00}=p_{11}=\tfrac12$, $p_{01}=p_{10}=0$. With $N$ shots the counts are **multinomial**:
$$(C_x)_x\sim\text{Multinomial}(N,(p_x)).$$
For a single outcome, $C_x\sim\text{Binomial}(N,p_x)$, and the empirical frequency $\hat p_x=C_x/N$ satisfies
$$\mathbb E[\hat p_x]=p_x\ \text{(unbiased)},\qquad \operatorname{Var}[\hat p_x]=\frac{p_x(1-p_x)}{N},\qquad \text{std error}\sim\frac1{\sqrt N}.$$
So **more shots → more accuracy**, but only as $1/\sqrt N$: a $10\times$ better estimate costs $100\times$ the shots. Let's watch it happen.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, N in zip(axes, [8, 64, 1024]):
    c = StatevectorSampler(seed=1).run([bell_meas], shots=N).result()[0].data.meas.get_counts()
    plot_histogram(c, ax=ax)
    ax.set_title(f"{N} shots")
plt.tight_layout(); plt.show()

At 8 shots the bars are lopsided and stray outcomes can appear by chance; by 1024 they sit near the ideal 50/50. To quantify "accuracy" we use the **total variation distance** to the ideal distribution,
$$\text{TVD}=\tfrac12\sum_x|\hat p_x-p_x|,$$
which is $0$ for a perfect match. We expect TVD $\sim 1/\sqrt N$.

In [ ]:
ideal = {"00": 0.5, "11": 0.5, "01": 0.0, "10": 0.0}

def tvd(shots, seed):
    c = StatevectorSampler(seed=seed).run([bell_meas], shots=shots).result()[0].data.meas.get_counts()
    f = {k: c.get(k, 0) / shots for k in ideal}
    return 0.5 * sum(abs(f[k] - ideal[k]) for k in ideal)

Ns = np.array([8, 32, 128, 512, 2048, 8192, 32768])
reps = 40
mean_tvd = np.array([np.mean([tvd(N, r) for r in range(reps)]) for N in Ns])
ref = mean_tvd[0] * np.sqrt(Ns[0]) / np.sqrt(Ns)     # 1/sqrt(N), anchored at first point

plt.figure(figsize=(7, 5))
plt.loglog(Ns, mean_tvd, "o-", label="measured mean TVD")
plt.loglog(Ns, ref, "--", label=r"reference $\propto 1/\sqrt{N}$")
plt.xlabel("number of shots  N"); plt.ylabel("distance from ideal (TVD)")
plt.title("Bell state: sampling accuracy improves like 1/sqrt(N)")
plt.legend(); plt.grid(True, which="both", alpha=0.3); plt.show()

The measured curve runs parallel to the $1/\sqrt N$ reference — a slope of $-\tfrac12$ on log–log axes. That is the multinomial standard error, seen directly: **accuracy grows with shots, but with diminishing returns.**

## Summary

1. **Gates & circuits** — qubits are unit vectors in $\mathbb C^2$; gates are unitary matrices; $|amp|^2$ gives measurement probabilities (Born rule).
2. **Bell state & entanglement** — `H`+`CX` builds $\tfrac1{\sqrt2}(|00\rangle+|11\rangle)$, a state that **cannot be factored**: random alone, perfectly correlated together.
3. **Transpilation** — the same Bell circuit, compiled for **two backends**, comes out differently: the **translation** stage rewrites gates into each device's native basis (CX-native stayed shallow; CZ-native grew). Stages run in order *init → layout → routing → translation → optimization*, each a group of passes.
4. **Sampling & shots** — counts are **multinomial**; the empirical frequency is an unbiased estimate of $p$, and accuracy improves only as $1/\sqrt N$.

## Take Home Questions

Now that you've learned how to create the Bell state |Φ⁺⟩ = (|00⟩ + |11⟩)/√2, it's time to practice creating the other three Bell states!

### The Four Bell States

There are four maximally entangled Bell states:

1. **|Φ⁺⟩ = (|00⟩ + |11⟩)/√2** ← We created this one!
2. **|Φ⁻⟩ = (|00⟩ - |11⟩)/√2**
3. **|Ψ⁺⟩ = (|01⟩ + |10⟩)/√2**
4. **|Ψ⁻⟩ = (|01⟩ - |10⟩)/√2**

### Your Task

For each of the remaining three Bell states:

1. **Create the quantum circuit** that produces the state
   - Hint: Start with H and CX gates (like we did for |Φ⁺⟩)
   - Think about what additional gates you might need (X, Z, etc.)

2. **Sample from the circuit** using the simulator
   - Use at least 1000 shots
   - Create a histogram of the results

3. **Verify your results**
   - Which measurement outcomes should you see?
   - What should their probabilities be?

### Hints

- **For |Φ⁻⟩**: You need to introduce a relative phase of -1 between |00⟩ and |11⟩
- **For |Ψ⁺⟩**: You need to flip one of the qubits before or after entanglement
- **For |Ψ⁻⟩**: Combine the ideas from the previous two hints

### Example Structure

```python
# Create circuit for |Φ⁻⟩
qc_phi_minus = QuantumCircuit(2)
# Add your gates here
# ...
qc_phi_minus.measure_all()

# Sample and visualize
job = sampler.run([qc_phi_minus], shots=1000)
counts = job.result()[0].data.meas.get_counts()
plot_histogram(counts, title="Bell state |Φ⁻⟩")
```

Good luck! These exercises will deepen your understanding of entanglement and quantum gates.